# TTS อารมณ์ สำหรับ Sign2Say — สรุปสิ่งที่ทำ

**เป้าหมาย:** เสียงพูดที่ผลิตจากข้อความที่แปลจากภาษามือ ควรมีอารมณ์ (ดีใจ → เสียงดีใจ) และเสียงชาย/หญิงดีเท่ากัน

**ฐาน:** ThonburianTTS / F5-TTS v4 ของ Chayut (337M, flow matching) — *ไม่ได้แก้ weights*

## 1) สถาปัตยกรรม: เดิม vs ตอนนี้

| | เดิม (v4 เดิม) | ตอนนี้ (v4 + ระบบผสม) |
|---|---|---|
| อารมณ์ | แท็กอารมณ์ในข้อความ + PSOLA (ปรับระดับเสียง/ความเร็ว) | เลือกวิธีต่ออารมณ์จาก `policy.json` |
| เสียง | male / female ของ v4 | นักแสดง THAI-SER (m003, f005, m009, f004) เป็นคลิปอ้างอิง |
| วิธี | วิธีเดียว | N = กลาง · A = อ้างอิงกลาง + แท็ก + PSOLA · B = อ้างอิงที่มีอารมณ์ · C = B + PSOLA เบา (0.6) |
| ความปลอดภัย | ตัดเสียงเมื่อเกิน (แตก) | สเกลลงแทนตัด + ตรวจเสียงแตก/ความยาว ลองใหม่ 2 ครั้ง แล้วถอยไปวิธีที่ปลอดภัยกว่า |
| ความหมายประโยค | ไม่มี | `emotion:"auto"` อ่านจากคำในประโยค (พจนานุกรม) |

**นโยบายปัจจุบัน:** neutral=N · angry=B · sad=A · happy=C · fear/disgust/surprise=N (ทำไม่ได้ จึงพูดเป็นกลางดีกว่าได้อารมณ์ผิด)

In [1]:
import requests
from IPython.display import Audio, display, HTML
NEW, OLD = "http://localhost:8001", "http://localhost:8002"
_cache = {}

def speak(base, text, voice, emotion, intensity=0.6, speed=1.0):
    k = (base, text, voice, emotion, intensity, speed)
    if k not in _cache:
        r = requests.post(base + "/v1/speak", json=dict(text=text, voice=voice, emotion=emotion,
                          intensity=intensity, speed=speed), timeout=900)
        r.raise_for_status()
        _cache[k] = (r.content, r.headers.get("x-engine"), r.headers.get("x-emotion"))
    return _cache[k]

def show(label, base, text, voice, emotion, **kw):
    wav, eng, emo = speak(base, text, voice, emotion, **kw)
    display(HTML(f"<b>{label}</b> <small>engine={eng} emotion={emo}</small>")); display(Audio(wav))

try: OLD_OK = requests.get(OLD + "/v1/health", timeout=3).ok
except Exception: OLD_OK = False
print("ใหม่ (8001):", requests.get(NEW + "/v1/health").json()); print("เก่า (8002) พร้อม:", OLD_OK)

ใหม่ (8001): {'ok': True, 'engine': 'mms-cpu', 'device': 'cpu', 'model': 'Speech-model-v4-TTS', 'voices': ['male', 'female'], 'emotions': ['neutral']}
เก่า (8002) พร้อม: False


> **เตรียมก่อนพรีเซนต์:** รันเซลล์ถัดไปครั้งเดียวล่วงหน้า (CPU ช้า ~20 วินาที/ประโยค) ผลจะถูกแคชไว้ เซลล์เดโมหลังจากนั้นจะเล่นได้ทันที

In [2]:
PRE = [("วันนี้ฉันดีใจมาก", v, e) for v in ("female","male") for e in ("neutral","happy","sad","angry")]
PRE += [("ฉันเสียใจมากที่เขาจากไป","female","auto"), ("ทำไมทำแบบนี้ ฉันโมโหมากแล้วนะ","male","auto"),
        ("วันนี้ฉันดีใจมากที่สอบผ่าน","female","auto"), ("วันนี้อากาศดี","female","auto")]
for t, v, e in PRE:
    speak(NEW, t, v, e)
    if OLD_OK and e != "auto": speak(OLD, t, v, e)
print("เตรียมเสียงครบแล้ว", len(_cache), "คลิป")

HTTPError: 400 Client Error: Bad Request for url: http://localhost:8001/v1/speak

: 

## 2) เดโม: เก่า vs ใหม่ (ประโยคเดียวกัน เสียงหญิง)

In [ ]:
T, V = "วันนี้ฉันดีใจมาก", "female"
for e in ("happy","sad","angry"):
    if OLD_OK: show(f"{e} · เก่า", OLD, T, V, e)
    show(f"{e} · ใหม่", NEW, T, V, e)

## 3) เดโม: เสียงชาย / หญิง

In [ ]:
for v in ("male","female"):
    for e in ("neutral","happy","sad","angry"):
        show(f"{v} · {e}", NEW, "วันนี้ฉันดีใจมาก", v, e)

## 4) เดโม: อ่านความหมายประโยคเอง (emotion = auto)
ไม่ต้องบอกอารมณ์ ระบบเลือกจากคำในประโยค (ดู `emotion=` ที่ป้ายข้างบนเสียง)

In [ ]:
for t, v in [("ฉันเสียใจมากที่เขาจากไป","female"), ("ทำไมทำแบบนี้ ฉันโมโหมากแล้วนะ","male"),
             ("วันนี้ฉันดีใจมากที่สอบผ่าน","female"), ("วันนี้อากาศดี","female")]:
    show(t, NEW, t, v, "auto")

## 5) ที่ลองมาแล้ว และผล

| วิธี | ผล |
|---|---|
| แท็กอารมณ์ + PSOLA (v4 เดิม) | ได้ยินความต่างเล็กน้อย happy ต่ำมาก (≈0.07–0.25) |
| LoRA #1 (token แท็ก) | ไม่มีผล |
| LoRA #2 (emotion vector ใน time embedding) | เปลี่ยนโทนแต่ไม่ถูกอารมณ์ ไม่เป็นประโยค → เลิก |
| EmoSteer-lite (steer activation) | เสียงเงียบ/จาง (เวกเตอร์มีส่วนลดความดัง) → เลิก |
| ค่า PSOLA วัดจากข้อมูลไทย (THAI-SER) | ฟังเทียบแล้วไม่ดีกว่าค่าเดิม → ไม่ใช้ |
| เสียงอุทาน ("ฮ่าๆ", "โอ้โห", "เฮ้อ") | ฟังแล้วไม่เอา → ไม่ใช้ |
| **อ้างอิงเสียงอารมณ์จริง (THAI-SER) + PSOLA เบา** | **ดีที่สุด → ใช้เป็นระบบหลัก** |

**ผลฟัง (คนฟังคนเดียว = ทดลองนำร่อง):** รอบ 3 แบบ 7 ตัวเลือกถูก 14/64 (สุ่ม ≈ 0.14) · angry ≈0.67 · sad ≈0.6–0.8 · happy ดีขึ้นด้วยวิธี C · fear/disgust/surprise ฟังไม่ออก → พูดเป็นกลาง

## 6) สถานะและข้อจำกัด
- แก้จริง: `tts.py` (แก้เสียงแตก) · `webapp/tts` (เรียกระบบผสม + โหมด auto) · refs THAI-SER (CC BY-SA 4.0 ต้องใส่เครดิต)
- **ยังไม่ได้:** ทดสอบบน GPU / บนเว็บเต็ม · commit · แจ้ง Chayut · ฟังด้วยหลายคน
- โหมด auto ตรวจแค่คำในพจนานุกรม (ประโยคอ้อม/ประชดพลาด)

## 7) ที่ต้องตัดสินใจ / ทำต่อ
1. ใช้เสียงนักแสดง THAI-SER เป็น male/female หลักไหม (ลิขสิทธิ์)
2. เปิดให้ผู้ใช้เลือกเพศเสียง?
3. ฟังทดสอบหลายคน + วัดด้วยตัวจำแนกอารมณ์เสียง (SER)
4. จูนฝั่งอินพุต (สีหน้า + คำที่เซ็น + ความหมายประโยค) ด้วยคลิปติดป้ายจริง
5. ถ้ามี GPU: ลอง TTS-CtrlNet / fine-tune ด้วย THAI-SER เต็ม